# Duka Electronics Sales Exploration
**Author:** Buhle Pearl Nkosi · 1Million Devs Cohort · Week 3 Assignment

## Executive Summary
1. Duka Electronics recorded 1,000 orders worth KES 71,792,500 between 01 Jan 2026 and 29 Jun 2026.
2. Nairobi is the top branch with KES 26,106,100, 36.4% of revenue, versus Mombasa at 19.5%.
3. Laptops generate KES 41,535,000 (57.9% of revenue), the largest of the five products.
4. Online orders number 320 (32.0% of orders) with an average of KES 71,207, against KES 70,759 in-store.
5. Premium-priced products (Laptop and Tablet) account for 84.8% of revenue from 435 orders.

In [1]:
import numpy as np
import pandas as pd

sales = pd.read_csv('duka_sales.csv', parse_dates=['order_date'])

## First-Five-Minutes Inspection
Observations are written beneath each output.

In [2]:
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


**head():** orders start at ID 1001 with dates in early 2026; each row is one order line with a fixed unit price per product.

In [3]:
sales.tail()

,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


**tail():** IDs run consecutively to 2000, so no order IDs are missing at the end.

In [4]:
sales.shape

(1000, 7)

**shape:** 1,000 rows and 7 columns.

In [5]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB


**info():** no nulls in any column, and `order_date` parsed correctly as datetime64.

In [6]:
sales.describe()

,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


**describe():** quantity ranges 1–5 and unit price KES 1,200–65,000, with no impossible values.

In [7]:
sales.nunique()

order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5
dtype: int64

**nunique():** 5 branches, 5 products, 3 channels, and 1,000 unique order IDs.

In [8]:
print(sales.branch.value_counts(normalize=True).round(3)); print(sales.channel.value_counts(normalize=True).round(3))

branch
Nairobi    0.376
Mombasa    0.186
Nakuru     0.162
Kisumu     0.159
Eldoret    0.117
Name: proportion, dtype: float64
channel
In-store       0.579
Online         0.320
Phone order    0.101
Name: proportion, dtype: float64


**Mix:** Nairobi dominates order volume (about 40%) and just over half of orders are in-store, with online about a third and phone orders a tenth.

In [9]:
print(sales.order_date.min(), sales.order_date.max())

2026-01-01 00:00:00 2026-06-29 00:00:00


**Period:** data covers 1 Jan 2026 to late June 2026 (180 days).

## Derived Columns

In [10]:
sales['revenue'] = sales['quantity'] * sales['unit_price']
sales['month'] = sales['order_date'].dt.month_name()
sales['weekday'] = sales['order_date'].dt.day_name()
sales['is_weekend'] = sales['order_date'].dt.dayofweek >= 5
sales['price_band'] = np.where(sales['unit_price'] >= 28000, 'Premium', 'Value')
for col in ['month','weekday','is_weekend','price_band']:
    print(sales[col].value_counts(), '\n')

month
March       186
January     173
April       167
June        162
February    159
May         153
Name: count, dtype: int64 

weekday
Friday       166
Sunday       145
Thursday     145
Monday       140
Wednesday    137
Tuesday      135
Saturday     132
Name: count, dtype: int64 

is_weekend
False    723
True     277
Name: count, dtype: int64 

price_band
Value      565
Premium    435
Name: count, dtype: int64 



Counts look sensible: 6 months, 7 weekdays, about 2/7 weekend days, and Premium covers Laptop and Tablet only.

## Task 3: Ten Manager Questions

In [11]:
print('1. Nairobi orders:', (sales.branch=='Nairobi').sum())
print('2. Online laptop orders:', ((sales.channel=='Online')&(sales['product']=='Laptop')).sum())
print('3. Mombasa+Kisumu revenue:', f"{sales[sales.branch.isin(['Mombasa','Kisumu'])].revenue.sum():,}")
display(sales.nlargest(10,'revenue')[['order_id','branch','product','revenue']])
print('5. Quantity >= 4:', (sales.quantity>=4).sum())
print('6. Phone-order revenue share %:', round(sales[sales.channel=='Phone order'].revenue.sum()/sales.revenue.sum()*100,1))
march = sales[sales.order_date.dt.month==3]
print('7. March orders:', len(march))
q = sales.query("channel=='Online' and product=='Laptop'")
print('8. query() matches Q2:', len(q)==((sales.channel=='Online')&(sales['product']=='Laptop')).sum())
print(sales[sales.channel.isin(['In-store','Online'])].groupby('channel').revenue.mean().round(0))
march.to_csv('march_orders.csv', index=False)

1. Nairobi orders: 376
2. Online laptop orders: 68
3. Mombasa+Kisumu revenue: 24,871,800


,order_id,branch,product,revenue
39,1040,Mombasa,Laptop,325000
74,1075,Nairobi,Laptop,325000
80,1081,Kisumu,Laptop,325000
96,1097,Nakuru,Laptop,325000
99,1100,Nairobi,Laptop,325000
133,1134,Kisumu,Laptop,325000
152,1153,Mombasa,Laptop,325000
157,1158,Eldoret,Laptop,325000
200,1201,Nairobi,Laptop,325000
246,1247,Nakuru,Laptop,325000


5. Quantity >= 4: 399
6. Phone-order revenue share %: 11.2
7. March orders: 186
8. query() matches Q2: True
channel
In-store    70759.0
Online      71207.0
Name: revenue, dtype: float64


**Answer:** results are printed above; Nairobi leads order counts, and the `.query()` version returns the same count as the boolean mask. March orders were exported to `march_orders.csv` (not committed).

## Task 4: Branch Performance

In [12]:
branch = sales.groupby('branch').agg(total_revenue=('revenue','sum'), orders=('order_id','count'),
        avg_order_value=('revenue','mean'), units=('quantity','sum')).sort_values('total_revenue', ascending=False)
branch['share_pct'] = (branch.total_revenue/branch.total_revenue.sum()*100).round(1)
branch.round(0)

,total_revenue,orders,avg_order_value,units,share_pct
branch,,,,,
Nairobi,26106100,376,69431.0,1107,36.0
Mombasa,14003700,186,75289.0,567,20.0
Nakuru,11970400,162,73891.0,495,17.0
Kisumu,10868100,159,68353.0,473,15.0
Eldoret,8844200,117,75591.0,370,12.0


**Answer:** Nairobi contributes the largest share of revenue, roughly double the second branch, in line with its 40% share of orders.

## Task 5: Channel and Product Comparison

In [13]:
print(sales.groupby('channel').revenue.agg(['count','mean']).round(0))
print((sales.groupby('product').revenue.sum()/sales.revenue.sum()*100).round(1).sort_values(ascending=False))

             count     mean
channel                    
In-store       579  70759.0
Online         320  71207.0
Phone order    101  79576.0
product
Laptop        57.9
Tablet        26.9
Phone         12.4
Headphones     1.9
Charger        0.9
Name: revenue, dtype: float64


**Answer:** average order value is similar across In-store and Online, so growing online volume adds revenue without shrinking basket size. Laptops lead product revenue because of their KES 65,000 price.

## Task 6: Premium Online Orders from Coast and Capital

In [14]:
out = sales[(sales.price_band=='Premium') & (sales.channel=='Online') & (sales.branch.isin(['Nairobi','Mombasa']))]
out.to_csv('outputs/premium_online_coast_capital.csv', index=False)
print(len(out), out.revenue.sum())

83 10218000


**Answer:** only Premium, Online orders from Nairobi and Mombasa were saved with `index=False` to `outputs/premium_online_coast_capital.csv`; count and revenue are printed above.

## Task 7: Monthly Branch Pivot and Excel Reconciliation

In [15]:
pv = sales.pivot_table(index='branch', columns='channel', values='revenue', aggfunc='sum', margins=True)
pv

channel,In-store,Online,Phone order,All
branch,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
All,40969200,22786100,8037200,71792500


**Answer:** these branch × channel totals match the PivotTable and SUMIFS table in `excel/duka_branch_summary.xlsx`.